# Módulo 03 · Aula 02 — Consultas Básicas

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Na aula anterior você construiu o *lugar* onde os dados moram. Agora vai começar a **fazer perguntas**.

SQL é uma linguagem **declarativa**: você descreve **o que quer**, não **como buscar**. O otimizador do banco decide o caminho. É uma mudança de mentalidade em relação ao Python — e é libertadora.

## O que você vai aprender aqui

| # | Tópico | Pergunta que responde |
|---|--------|------------------------|
| 1 | `SELECT` e expressões | "Quais colunas eu quero?" |
| 2 | `WHERE` | "Quais linhas me interessam?" |
| 3 | Ordem lógica de execução | "Por que meu apelido não funciona no `WHERE`?" |
| 4 | `ORDER BY`, `LIMIT`, `OFFSET` | "Em que ordem? Quantas?" |
| 5 | Agregações | "Quanto? Quantos? Qual a média?" |
| 6 | `GROUP BY` | "E por cidade? Por mês? Por canal?" |
| 7 | `HAVING` | "Só os grupos que passam de X" |
| 8 | `CASE WHEN` | "Classifique em faixas" |
| 9 | Funções de texto, número e data | Limpeza e formatação |

## ⚙️ Preparando o banco

A célula abaixo cria o banco da Aurora **em memória** e o popula com dados sintéticos: 6 categorias, 20 produtos, 30 clientes, 180 pedidos e ~340 itens.

A semente aleatória é fixa (`42`), então **todos os números que você vir são iguais aos deste manual**.

Ela também define as funções `sql(...)` e `ddl(...)` que já usamos na aula anterior.

> ▶️ **Execute esta célula primeiro.** Sem ela, nada mais funciona.

In [ ]:
import sqlite3
import random

# ═══════════════════════════════════════════════════════════════
#  Banco de treino da Aurora Comércio
#  Execute esta célula UMA VEZ, antes de qualquer outra.
#  Ela é idempotente: pode rodar de novo a qualquer momento.
# ═══════════════════════════════════════════════════════════════

CONN = sqlite3.connect(":memory:")
CONN.execute("PRAGMA foreign_keys = ON")

CONN.executescript("""
CREATE TABLE categorias (
    id           INTEGER PRIMARY KEY,
    nome         TEXT    NOT NULL UNIQUE,
    margem_alvo  REAL    NOT NULL DEFAULT 0.25 CHECK (margem_alvo BETWEEN 0 AND 1)
);

CREATE TABLE produtos (
    id           INTEGER PRIMARY KEY,
    sku          TEXT    NOT NULL UNIQUE,
    nome         TEXT    NOT NULL,
    categoria_id INTEGER NOT NULL REFERENCES categorias(id) ON DELETE RESTRICT,
    preco        REAL    NOT NULL CHECK (preco >= 0),
    custo        REAL    NOT NULL CHECK (custo >= 0),
    estoque      INTEGER NOT NULL DEFAULT 0 CHECK (estoque >= 0),
    ativo        INTEGER NOT NULL DEFAULT 1 CHECK (ativo IN (0,1))
);

CREATE TABLE clientes (
    id            INTEGER PRIMARY KEY,
    nome          TEXT NOT NULL,
    email         TEXT NOT NULL UNIQUE,
    cidade        TEXT NOT NULL,
    uf            TEXT NOT NULL CHECK (length(uf) = 2),
    segmento      TEXT NOT NULL DEFAULT 'varejo'
                       CHECK (segmento IN ('varejo','corporativo')),
    data_cadastro TEXT NOT NULL,
    telefone      TEXT
);

CREATE TABLE pedidos (
    id          INTEGER PRIMARY KEY,
    cliente_id  INTEGER NOT NULL REFERENCES clientes(id) ON DELETE RESTRICT,
    data_pedido TEXT    NOT NULL,
    status      TEXT    NOT NULL CHECK (status IN ('pago','pendente','cancelado')),
    canal       TEXT    NOT NULL CHECK (canal IN ('site','app','marketplace')),
    frete       REAL    NOT NULL DEFAULT 0 CHECK (frete >= 0)
);

CREATE TABLE itens_pedido (
    id             INTEGER PRIMARY KEY,
    pedido_id      INTEGER NOT NULL REFERENCES pedidos(id)  ON DELETE CASCADE,
    produto_id     INTEGER NOT NULL REFERENCES produtos(id) ON DELETE RESTRICT,
    quantidade     INTEGER NOT NULL CHECK (quantidade > 0),
    preco_unitario REAL    NOT NULL CHECK (preco_unitario >= 0),
    UNIQUE (pedido_id, produto_id)
);
""")

_CATEGORIAS = [(1, "Notebooks", 0.18), (2, "Monitores", 0.22), (3, "Periféricos", 0.38),
               (4, "Armazenamento", 0.30), (5, "Redes", 0.28), (6, "Áudio", 0.35)]

_PRODUTOS = [
    ("NB-DELL-15",  "Notebook Dell Inspiron 15",     1, 2599.90, 2120.00,  14),
    ("NB-ACER-N5",  "Notebook Acer Nitro 5",         1, 3299.00, 2780.00,   7),
    ("NB-LEN-IP3",  "Notebook Lenovo IdeaPad 3",     1, 2199.00, 1850.00,  22),
    ("NB-APPL-M2",  "MacBook Air M2",                1, 9499.00, 8300.00,   3),
    ("MO-LG-24UW",  "Monitor LG 24 UltraWide",       2, 1199.00,  920.00,  31),
    ("MO-SAM-ODY",  "Monitor Samsung Odyssey 27",    2, 1849.00, 1420.00,  12),
    ("MO-AOC-22",   "Monitor AOC 22 Full HD",        2,  749.00,  560.00,  45),
    ("PE-LOG-MX3",  "Mouse Logitech MX Master 3",    3,  549.00,  340.00,  88),
    ("PE-LOG-M170", "Mouse Logitech M170",           3,   89.90,   52.00, 240),
    ("PE-RED-K552", "Teclado Redragon K552",         3,  249.00,  150.00,  64),
    ("PE-LOG-C920", "Webcam Logitech C920",          3,  449.00,  290.00,  37),
    ("AU-HYP-CL2",  "Headset HyperX Cloud II",       6,  399.00,  255.00,  29),
    ("AU-JBL-T510", "Fone JBL Tune 510BT",           6,  229.00,  140.00,  73),
    ("AR-SSD-1TB",  "SSD NVMe 1TB Kingston",         4,  489.00,  360.00,  52),
    ("AR-SSD-480",  "SSD SATA 480GB Sandisk",        4,  229.00,  158.00,  96),
    ("AR-HD-2TB",   "HD Externo 2TB Seagate",        4,  549.00,  410.00,  18),
    ("AR-PEN-128",  "Pendrive 128GB Sandisk",        4,   79.90,   44.00, 180),
    ("RE-TPL-AX55", "Roteador TP-Link Archer AX55",  5,  699.00,  505.00,  26),
    ("RE-TPL-RE30", "Repetidor TP-Link RE305",       5,  229.00,  152.00,  41),
    ("RE-INT-AX20", "Placa de Rede Intel AX200",     5,  189.00,  124.00,  33),
]

_NOMES = ["Ana Costa", "Bruno Rocha", "Carla Dias", "Daniel Souza", "Elisa Martins",
          "Fábio Nunes", "Gustavo Reis", "Helena Prado", "Igor Batista", "Julia Andrade",
          "Lucas Moreira", "Maria Souza", "Nathalia Freitas", "Otávio Pinto",
          "Priscila Gomes", "Rafael Torres", "Sabrina Melo", "Thiago Barros",
          "Vanessa Lima", "William Cruz", "Beatriz Almeida", "Caio Ferreira",
          "Débora Ramos", "Eduardo Pires", "Fernanda Vieira", "Gabriel Mendes",
          "Isabela Rocha", "João Lima", "Karina Duarte", "Leonardo Castro"]

_CIDADES = [("Campinas", "SP"), ("São Paulo", "SP"), ("Sorocaba", "SP"),
            ("Ribeirão Preto", "SP"), ("Jundiaí", "SP"), ("Santos", "SP"),
            ("Belo Horizonte", "MG"), ("Uberlândia", "MG"), ("Curitiba", "PR"),
            ("Londrina", "PR"), ("Porto Alegre", "RS"), ("Florianópolis", "SC"),
            ("Rio de Janeiro", "RJ"), ("Niterói", "RJ"), ("Salvador", "BA"),
            ("Recife", "PE"), ("Fortaleza", "CE"), ("Brasília", "DF"),
            ("Goiânia", "GO"), ("Vitória", "ES")]

_rnd = random.Random(42)     # semente fixa = todos veem os mesmos números

CONN.executemany("INSERT INTO categorias VALUES (?,?,?)", _CATEGORIAS)
CONN.executemany(
    "INSERT INTO produtos (sku,nome,categoria_id,preco,custo,estoque,ativo) "
    "VALUES (?,?,?,?,?,?,1)", _PRODUTOS)

_acentos = str.maketrans("áéíóúãõâêôç", "aeiouaoaeoc")
_clientes = []
for _i, _nome in enumerate(_NOMES, 1):
    _cidade, _uf = _rnd.choice(_CIDADES)
    _login = _nome.split()[0].lower().translate(_acentos)
    _clientes.append((
        _i, _nome, f"{_login}{_i}@email.com", _cidade, _uf,
        "corporativo" if _rnd.random() < 0.25 else "varejo",
        f"2026-{_rnd.randint(1, 6):02d}-{_rnd.randint(1, 28):02d}",
        f"(19) 9{_rnd.randint(1000, 9999)}-{_rnd.randint(1000, 9999)}" if _rnd.random() < 0.6 else None,
    ))
CONN.executemany("INSERT INTO clientes VALUES (?,?,?,?,?,?,?,?)", _clientes)

_pedidos, _itens, _id_item = [], [], 0
# Os 3 últimos clientes ficam SEM pedido de propósito: toda base real tem
# gente que se cadastrou e nunca comprou, e você precisa saber encontrá-los.
for _pid in range(1, 181):
    _mes = _rnd.choices([5, 6, 7], weights=[2, 3, 4])[0]
    _pedidos.append((
        _pid, _rnd.randint(1, len(_NOMES) - 3),
        f"2026-{_mes:02d}-{_rnd.randint(1, 28):02d}",
        _rnd.choices(["pago", "pendente", "cancelado"], weights=[80, 12, 8])[0],
        _rnd.choices(["site", "app", "marketplace"], weights=[50, 30, 20])[0],
        _rnd.choice([0.0, 9.90, 19.90, 29.90]),
    ))
    _n_itens = _rnd.choices([1, 2, 3, 4], weights=[45, 30, 17, 8])[0]
    for _prod in _rnd.sample(range(1, len(_PRODUTOS) + 1), _n_itens):
        _id_item += 1
        _itens.append((
            _id_item, _pid, _prod,
            _rnd.choices([1, 2, 3, 5, 10], weights=[55, 22, 12, 7, 4])[0],
            round(_PRODUTOS[_prod - 1][3] * _rnd.choice([1.0, 1.0, 1.0, 0.95, 0.90]), 2),
        ))
CONN.executemany("INSERT INTO pedidos VALUES (?,?,?,?,?,?)", _pedidos)
CONN.executemany("INSERT INTO itens_pedido VALUES (?,?,?,?,?)", _itens)
CONN.commit()


# ── Funções auxiliares ───────────────────────────────────────
def _fmt(valor):
    if valor is None:
        return "NULL"
    if isinstance(valor, float):
        return f"{valor:,.2f}"
    if isinstance(valor, int):
        return f"{valor:,}"
    return str(valor)


def sql(consulta, parametros=(), limite=30):
    """Executa uma consulta e imprime o resultado formatado."""
    try:
        cursor = CONN.execute(consulta, parametros)
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")
        return None

    if cursor.description is None:
        CONN.commit()
        print(f"✅ OK — {cursor.rowcount} linha(s) afetada(s)" if cursor.rowcount >= 0 else "✅ OK")
        return None

    colunas = [d[0] for d in cursor.description]
    linhas = cursor.fetchall()
    total = len(linhas)
    linhas = linhas[:limite]
    if not linhas:
        print("(nenhuma linha)")
        return []

    texto = [[_fmt(v) for v in linha] for linha in linhas]
    numerica = [
        any(isinstance(l[i], (int, float)) for l in linhas)
        and all(isinstance(l[i], (int, float)) or l[i] is None for l in linhas)
        for i in range(len(colunas))
    ]
    larguras = [max(len(colunas[i]), max(len(l[i]) for l in texto))
                for i in range(len(colunas))]

    def borda(e, m, d):
        return e + m.join("─" * (w + 2) for w in larguras) + d

    print(borda("┌", "┬", "┐"))
    print("│ " + " │ ".join(c.ljust(w) for c, w in zip(colunas, larguras)) + " │")
    print(borda("├", "┼", "┤"))
    for linha in texto:
        print("│ " + " │ ".join(
            (v.rjust(w) if numerica[i] else v.ljust(w))
            for i, (v, w) in enumerate(zip(linha, larguras))) + " │")
    print(borda("└", "┴", "┘"))
    print(f"{total} linha(s)" + (f" — exibindo as {limite} primeiras" if total > limite else ""))
    return linhas


def ddl(script):
    """Executa um script com vários comandos."""
    try:
        CONN.executescript(script)
        CONN.commit()
        print("✅ Script executado")
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")


print("✅ Banco da Aurora criado em memória\n")
sql("""
SELECT 'categorias'   AS tabela, COUNT(*) AS linhas FROM categorias
UNION ALL SELECT 'produtos',     COUNT(*) FROM produtos
UNION ALL SELECT 'clientes',     COUNT(*) FROM clientes
UNION ALL SELECT 'pedidos',      COUNT(*) FROM pedidos
UNION ALL SELECT 'itens_pedido', COUNT(*) FROM itens_pedido
""")

## 1. `SELECT` — a forma mais simples

```sql
SELECT colunas
FROM tabela;
```

O `*` traz todas as colunas. É ótimo para explorar e **ruim em produção**: traz dados que você não usa, quebra quando alguém adiciona uma coluna, e esconde a intenção.

In [ ]:
sql("SELECT * FROM produtos LIMIT 5")

In [ ]:
# Colunas específicas — sempre prefira isto em código de verdade
sql("SELECT sku, nome, preco, estoque FROM produtos LIMIT 5")

### Expressões e apelidos (`AS`)

Você pode calcular na consulta. Use `AS` para nomear o resultado — sem isso a coluna vira a própria expressão, ilegível.

In [ ]:
sql("""
SELECT
    nome,
    preco,
    custo,
    preco - custo                      AS margem_reais,
    ROUND((preco - custo) / preco, 4)  AS margem_pct,
    preco * estoque                    AS valor_em_estoque
FROM produtos
LIMIT 8
""")

### ⚠️ Divisão inteira

No SQLite (como em muitos bancos), dividir dois **inteiros** devolve um **inteiro**. `5 / 2` é `2`, não `2.5`.

A correção: force um dos operandos a ser real, multiplicando por `1.0` ou usando `CAST`.

In [ ]:
sql("""
SELECT
    5 / 2               AS "5/2 (inteiro)",
    5.0 / 2             AS "5.0/2",
    5 * 1.0 / 2         AS "5*1.0/2",
    CAST(5 AS REAL) / 2 AS "CAST(5 AS REAL)/2"
""")

### `||` — concatenação

No SQL padrão (e no SQLite) o operador de concatenação é `||`, não `+`.

In [ ]:
sql("""
SELECT
    nome || ' (' || sku || ')'          AS identificacao,
    'R$ ' || printf('%.2f', preco)      AS preco_formatado,
    upper(substr(nome, 1, 3))           AS prefixo
FROM produtos
LIMIT 5
""")

### `DISTINCT` — removendo duplicatas

In [ ]:
sql("SELECT DISTINCT uf FROM clientes ORDER BY uf")

In [ ]:
# DISTINCT vale para a COMBINAÇÃO de todas as colunas listadas
sql("SELECT DISTINCT uf, segmento FROM clientes ORDER BY uf, segmento LIMIT 12")

## 2. `WHERE` — filtrando linhas

| Operador | Uso | Exemplo |
|----------|-----|---------|
| `=` `<>` `!=` | Igualdade | `status = 'pago'` |
| `>` `<` `>=` `<=` | Comparação | `preco > 1000` |
| `AND` `OR` `NOT` | Lógicos | `preco > 1000 AND estoque < 10` |
| `BETWEEN a AND b` | Intervalo (**inclusivo**) | `preco BETWEEN 100 AND 500` |
| `IN (...)` | Pertence a um conjunto | `uf IN ('SP','RJ','MG')` |
| `LIKE` | Padrão de texto | `nome LIKE 'Notebook%'` |
| `IS NULL` / `IS NOT NULL` | Ausência de valor | `telefone IS NULL` |

In [ ]:
sql("SELECT sku, nome, preco, estoque FROM produtos WHERE preco > 1000 ORDER BY preco DESC")

In [ ]:
# AND / OR — use parênteses, sempre. O AND tem precedência sobre o OR.
sql("""
SELECT sku, nome, preco, estoque
FROM produtos
WHERE (categoria_id = 1 OR categoria_id = 2)
  AND estoque < 20
ORDER BY estoque
""")

> ⚠️ **Sem parênteses, `A OR B AND C` é lido como `A OR (B AND C)`.** Isso já causou muito relatório errado. Escreva os parênteses mesmo quando forem redundantes — eles custam nada e documentam sua intenção.

In [ ]:
# BETWEEN é INCLUSIVO nos dois extremos
sql("""
SELECT sku, nome, preco
FROM produtos
WHERE preco BETWEEN 200 AND 500
ORDER BY preco
""")

In [ ]:
# IN — muito mais legível que uma cadeia de OR
sql("""
SELECT nome, cidade, uf, segmento
FROM clientes
WHERE uf IN ('SP', 'RJ', 'MG')
ORDER BY uf, cidade
LIMIT 12
""")

### `LIKE` — buscando por padrão

| Curinga | Significa |
|---------|-----------|
| `%` | Zero ou mais caracteres |
| `_` | Exatamente um caractere |

| Padrão | Encontra |
|--------|----------|
| `'Note%'` | Começa com "Note" |
| `'%Dell%'` | Contém "Dell" |
| `'%15'` | Termina com "15" |
| `'NB-____-__'` | "NB-" + 4 chars + "-" + 2 chars |

In [ ]:
sql("SELECT sku, nome FROM produtos WHERE nome LIKE 'Notebook%'")

In [ ]:
sql("SELECT sku, nome FROM produtos WHERE nome LIKE '%Logitech%'")

In [ ]:
sql("SELECT sku, nome FROM produtos WHERE sku LIKE 'AR-%'")

### ⚠️ `LIKE` e maiúsculas: a pegadinha dos acentos

No SQLite, o `LIKE` é **insensível a maiúsculas — mas só para caracteres ASCII**. `'a'` casa com `'A'`, porém `'á'` **não** casa com `'Á'`.

Isso pega desprevenido em português. Se a busca precisa ignorar caixa de forma confiável, normalize os dois lados com `lower()`.

In [ ]:
sql("""
SELECT
    'Notebook' LIKE 'notebook'  AS "ASCII: casa?",
    'Áudio'    LIKE 'áudio'     AS "Acento: casa?",
    lower('Áudio') LIKE lower('ÁUDIO') AS "Com lower(): casa?"
""")

In [ ]:
# Busca robusta: normalize os dois lados
sql("""
SELECT sku, nome
FROM produtos
WHERE lower(nome) LIKE lower('%LOGITECH%')
""")

> 💡 **E o `_` literal?** Se você precisa buscar um sublinhado ou um `%` de verdade, use `ESCAPE`:
> ```sql
> WHERE codigo LIKE '%\_%' ESCAPE '\'
> ```

### `IS NULL` — o filtro que não usa `=`

Lembre da aula anterior: `NULL = NULL` não é verdadeiro. Para testar ausência, existe `IS NULL`.

In [ ]:
print("① Errado (= NULL nunca casa):")
sql("SELECT COUNT(*) AS encontrados FROM clientes WHERE telefone = NULL")

print("\n② Certo:")
sql("SELECT COUNT(*) AS sem_telefone FROM clientes WHERE telefone IS NULL")

print("\n③ Com telefone:")
sql("SELECT COUNT(*) AS com_telefone FROM clientes WHERE telefone IS NOT NULL")

In [ ]:
# ⚠️ NOT IN com NULL na lista: armadilha clássica
sql("""
SELECT
    3 IN (1, 2, 3)          AS "3 IN (1,2,3)",
    3 NOT IN (1, 2)         AS "3 NOT IN (1,2)",
    3 NOT IN (1, 2, NULL)   AS "3 NOT IN (1,2,NULL)"
""")

> 🔴 **`3 NOT IN (1, 2, NULL)` devolve `NULL`, não `1`.**
>
> Faz sentido logicamente: "3 é diferente de 1, e de 2, e de *desconhecido*?" — não dá para afirmar. E como `NULL` não é verdadeiro, a linha é **descartada** pelo `WHERE`.
>
> Consequência prática: um `NOT IN (SELECT ...)` onde a subconsulta devolve algum `NULL` retorna **zero linhas** silenciosamente. É um dos bugs mais difíceis de achar em SQL. Prefira `NOT EXISTS` (aula 03_03) ou adicione `WHERE coluna IS NOT NULL` na subconsulta.

## 3. A ordem lógica de execução

O SQL é escrito em uma ordem e **executado em outra**. Entender isso resolve metade das dúvidas.

```
   Ordem em que você ESCREVE      Ordem em que o banco EXECUTA
   ────────────────────────       ─────────────────────────────
   SELECT     ....... 5           1.  FROM      (de onde vêm as linhas)
   FROM       ....... 1           2.  WHERE     (filtra LINHAS)
   WHERE      ....... 2           3.  GROUP BY  (agrupa)
   GROUP BY   ....... 3           4.  HAVING    (filtra GRUPOS)
   HAVING     ....... 4           5.  SELECT    (calcula as colunas)
   ORDER BY   ....... 6           6.  ORDER BY  (ordena)
   LIMIT      ....... 7           7.  LIMIT     (corta)
```

**Duas consequências práticas que explicam quase todos os erros de iniciante:**

1. **O `WHERE` roda ANTES do `SELECT`.** Por isso, na maioria dos bancos, você não pode usar um apelido do `SELECT` dentro do `WHERE` — quando o `WHERE` executa, o apelido ainda não existe.
2. **O `ORDER BY` roda DEPOIS do `SELECT`.** Por isso ele **pode** usar apelidos.

In [ ]:
# O ORDER BY enxerga o apelido, porque roda depois do SELECT
sql("""
SELECT sku, nome, preco - custo AS margem
FROM produtos
ORDER BY margem DESC
LIMIT 5
""")

In [ ]:
# ⚠️ O SQLite é PERMISSIVO e aceita apelido no WHERE.
#    PostgreSQL, SQL Server e Oracle REJEITAM. Não crie o hábito.
print("① Funciona no SQLite, quebra no PostgreSQL:")
sql("SELECT sku, preco - custo AS margem FROM produtos WHERE margem > 500 LIMIT 3")

print("\n② Portável — repete a expressão:")
sql("SELECT sku, preco - custo AS margem FROM produtos WHERE preco - custo > 500 LIMIT 3")

## 4. `ORDER BY`, `LIMIT` e `OFFSET`

```sql
ORDER BY coluna [ASC | DESC], outra_coluna [ASC | DESC]
LIMIT n OFFSET m
```

- `ASC` (crescente) é o padrão.
- Você pode ordenar por várias colunas — desempate da esquerda para a direita.
- `LIMIT` + `OFFSET` é a base da **paginação** (que você vai reencontrar no Módulo 06, na API).

In [ ]:
sql("""
SELECT nome, cidade, uf, segmento
FROM clientes
ORDER BY uf ASC, cidade ASC, nome ASC
LIMIT 10
""")

In [ ]:
# Paginação: página 1 e página 2 de 5 em 5
print("── Página 1 ──")
sql("SELECT id, sku, preco FROM produtos ORDER BY preco DESC LIMIT 5 OFFSET 0")
print("\n── Página 2 ──")
sql("SELECT id, sku, preco FROM produtos ORDER BY preco DESC LIMIT 5 OFFSET 5")

> ⚠️ **`OFFSET` fica lento em tabelas grandes.** Para pular 100.000 linhas, o banco precisa ler e descartar as 100.000. Em produção usa-se **paginação por cursor** (`WHERE id > ultimo_id LIMIT 20`), que você vai ver no Módulo 07.
>
> ⚠️ **Ordenação e `NULL`.** No SQLite, `NULL` vem **primeiro** em `ASC` e por último em `DESC`. Outros bancos fazem o contrário. Se importa, seja explícito: `ORDER BY coluna DESC NULLS LAST`.

In [ ]:
sql("""
SELECT nome, telefone
FROM clientes
ORDER BY telefone
LIMIT 6
""")

## 5. Funções de agregação

Elas colapsam **muitas linhas em um valor**.

| Função | Devolve |
|--------|---------|
| `COUNT(*)` | Número de **linhas** |
| `COUNT(coluna)` | Número de valores **não nulos** naquela coluna |
| `COUNT(DISTINCT coluna)` | Valores distintos não nulos |
| `SUM(coluna)` | Soma |
| `AVG(coluna)` | Média |
| `MIN` / `MAX` | Menor / maior |
| `GROUP_CONCAT(coluna, ', ')` | Junta os valores numa string |
| `TOTAL(coluna)` | Como `SUM`, mas devolve `0.0` em vez de `NULL` (SQLite) |

In [ ]:
sql("""
SELECT
    COUNT(*)                  AS produtos,
    ROUND(AVG(preco), 2)      AS preco_medio,
    MIN(preco)                AS mais_barato,
    MAX(preco)                AS mais_caro,
    SUM(estoque)              AS itens_em_estoque,
    ROUND(SUM(preco * estoque), 2) AS valor_imobilizado
FROM produtos
""")

### ⚠️ `COUNT(*)` vs `COUNT(coluna)` — a diferença que muda o número

**Agregações ignoram `NULL`.** `COUNT(*)` conta linhas; `COUNT(coluna)` conta valores presentes.

In [ ]:
sql("""
SELECT
    COUNT(*)                  AS total_clientes,
    COUNT(telefone)           AS com_telefone,
    COUNT(*) - COUNT(telefone) AS sem_telefone,
    COUNT(DISTINCT cidade)    AS cidades_distintas,
    COUNT(DISTINCT uf)        AS ufs_distintas
FROM clientes
""")

In [ ]:
# AVG também ignora NULL — o denominador muda!
sql("""
SELECT
    COUNT(*)                                  AS linhas,
    COUNT(telefone)                           AS telefones_preenchidos,
    AVG(CASE WHEN telefone IS NULL THEN 0 ELSE 1 END) AS taxa_preenchimento
FROM clientes
""")

> 🔴 **A armadilha do `AVG` com `NULL`.** Se você calcula a nota média de 100 alunos e 30 não fizeram a prova (`NULL`), `AVG(nota)` divide por **70**, não por 100. Isso pode ser o que você quer — ou um erro grave. Decida conscientemente e, se precisar contar os ausentes como zero, use `AVG(COALESCE(nota, 0))`.

In [ ]:
# SUM sem linhas devolve NULL, não 0 — proteja com COALESCE
sql("""
SELECT
    SUM(preco)                     AS soma_sem_linhas,
    COALESCE(SUM(preco), 0)        AS com_coalesce,
    TOTAL(preco)                   AS com_total_sqlite,
    COUNT(*)                       AS linhas
FROM produtos
WHERE preco > 999999
""")

## 6. `GROUP BY` — agregando por categoria

```sql
SELECT coluna_de_grupo, AGREGACAO(...)
FROM tabela
GROUP BY coluna_de_grupo;
```

**A regra de ouro:** toda coluna do `SELECT` que **não** está dentro de uma função de agregação precisa estar no `GROUP BY`.

In [ ]:
sql("""
SELECT
    uf,
    COUNT(*)                AS clientes,
    COUNT(DISTINCT cidade)  AS cidades
FROM clientes
GROUP BY uf
ORDER BY clientes DESC, uf
""")

In [ ]:
# Agrupando por mais de uma coluna
sql("""
SELECT
    uf,
    segmento,
    COUNT(*) AS clientes
FROM clientes
GROUP BY uf, segmento
ORDER BY uf, segmento
LIMIT 15
""")

### ⚠️ A permissividade do SQLite (de novo)

Se você colocar uma coluna no `SELECT` sem incluí-la no `GROUP BY`, o **PostgreSQL dá erro**. O SQLite e o MySQL antigo aceitam e devolvem **um valor arbitrário** daquele grupo — silenciosamente errado.

In [ ]:
# O SQLite aceita. O 'nome' é de UM cliente qualquer do grupo — provavelmente não o que você quer.
sql("""
SELECT uf, nome, COUNT(*) AS clientes
FROM clientes
GROUP BY uf
ORDER BY clientes DESC
LIMIT 5
""")

> 🧭 **Regra de sobrevivência:** escreva SQL como se você estivesse no PostgreSQL. Toda coluna não agregada vai para o `GROUP BY`. Assim seu SQL do M03 continua funcionando no M05, quando migrarmos o Atlas para Postgres.

In [ ]:
# Agora a pergunta de negócio: faturamento por status de pedido
sql("""
SELECT
    status,
    COUNT(*)                        AS pedidos,
    ROUND(AVG(frete), 2)            AS frete_medio,
    ROUND(SUM(frete), 2)            AS frete_total
FROM pedidos
GROUP BY status
ORDER BY pedidos DESC
""")

In [ ]:
# Faturamento por canal, usando a tabela de itens
sql("""
SELECT
    p.canal,
    COUNT(DISTINCT p.id)                           AS pedidos,
    SUM(i.quantidade)                              AS itens,
    ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS faturamento,
    ROUND(SUM(i.quantidade * i.preco_unitario) / COUNT(DISTINCT p.id), 2) AS ticket_medio
FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
WHERE p.status = 'pago'
GROUP BY p.canal
ORDER BY faturamento DESC
""")

> 💡 **Repare no `COUNT(DISTINCT p.id)`.** Depois do `JOIN`, cada pedido aparece **uma vez por item**. Um `COUNT(*)` contaria itens, não pedidos. Esse é o erro mais comum ao agregar sobre tabelas juntadas — e o motivo pelo qual relatórios "inflam" os números.

## 7. `HAVING` — filtrando **grupos**

| | `WHERE` | `HAVING` |
|---|---------|----------|
| Filtra | **Linhas** individuais | **Grupos** já agregados |
| Roda | Antes do `GROUP BY` | Depois do `GROUP BY` |
| Pode usar agregações? | ❌ Não | ✅ Sim |

**Regra prática:** se a condição usa `COUNT`, `SUM`, `AVG`... vai no `HAVING`. Caso contrário, vai no `WHERE` — e é mais eficiente, porque descarta linhas antes de agrupar.

In [ ]:
# Cidades com mais de um cliente
sql("""
SELECT cidade, uf, COUNT(*) AS clientes
FROM clientes
GROUP BY cidade, uf
HAVING COUNT(*) > 1
ORDER BY clientes DESC, cidade
""")

In [ ]:
# WHERE e HAVING juntos — cada um no seu papel
sql("""
SELECT
    c.uf,
    COUNT(DISTINCT p.id)                           AS pedidos,
    ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS faturamento
FROM pedidos p
JOIN clientes c     ON c.id = p.cliente_id
JOIN itens_pedido i ON i.pedido_id = p.id
WHERE p.status = 'pago'                    -- filtra LINHAS antes de agrupar
GROUP BY c.uf
HAVING SUM(i.quantidade * i.preco_unitario) > 50000   -- filtra GRUPOS depois
ORDER BY faturamento DESC
""")

## 8. `CASE WHEN` — o `if/else` do SQL

```sql
CASE
    WHEN condição1 THEN resultado1
    WHEN condição2 THEN resultado2
    ELSE resultado_padrao
END
```

É a ferramenta mais versátil do SQL. Serve para classificar, criar colunas derivadas e — o truque mais útil — **pivotar** dados.

In [ ]:
sql("""
SELECT
    sku,
    nome,
    preco,
    CASE
        WHEN preco >= 3000 THEN 'Premium'
        WHEN preco >= 1000 THEN 'Intermediário'
        WHEN preco >=  300 THEN 'Popular'
        ELSE 'Entrada'
    END AS faixa,
    CASE
        WHEN estoque = 0  THEN '[!] esgotado'
        WHEN estoque < 10 THEN '[*] critico'
        WHEN estoque < 40 THEN '[ ] normal'
        ELSE                   '[+] excesso'
    END AS situacao_estoque
FROM produtos
ORDER BY preco DESC
LIMIT 12
""")

In [ ]:
# CASE + GROUP BY: distribuição por faixa
sql("""
SELECT
    CASE
        WHEN preco >= 3000 THEN 'Premium'
        WHEN preco >= 1000 THEN 'Intermediário'
        WHEN preco >=  300 THEN 'Popular'
        ELSE 'Entrada'
    END AS faixa,
    COUNT(*)                       AS produtos,
    ROUND(AVG(preco), 2)           AS preco_medio,
    SUM(estoque)                   AS estoque_total
FROM produtos
GROUP BY faixa
ORDER BY preco_medio DESC
""")

### O truque do pivô: `SUM(CASE WHEN ...)`

Para transformar **linhas em colunas** — a "tabela dinâmica" do Excel — combine agregação com `CASE`.

In [ ]:
sql("""
SELECT
    c.uf,
    COUNT(*)                                                  AS pedidos_total,
    SUM(CASE WHEN p.status = 'pago'      THEN 1 ELSE 0 END)    AS pagos,
    SUM(CASE WHEN p.status = 'pendente'  THEN 1 ELSE 0 END)    AS pendentes,
    SUM(CASE WHEN p.status = 'cancelado' THEN 1 ELSE 0 END)    AS cancelados,
    ROUND(100.0 * SUM(CASE WHEN p.status = 'cancelado' THEN 1 ELSE 0 END) / COUNT(*), 1)
        AS pct_cancelamento
FROM pedidos p
JOIN clientes c ON c.id = p.cliente_id
GROUP BY c.uf
ORDER BY pedidos_total DESC
LIMIT 12
""")

> 💡 **`SUM(CASE WHEN cond THEN 1 ELSE 0 END)`** é o idioma para "conte quantos satisfazem a condição, dentro de cada grupo". É provavelmente o padrão mais útil de todo o SQL analítico — grave-o.
>
> Variação: `COUNT(CASE WHEN cond THEN 1 END)` faz o mesmo (sem `ELSE`, o resultado é `NULL` e o `COUNT` ignora).

## 9. Funções úteis de texto, número e data

### Texto

| Função | Faz |
|--------|-----|
| `upper(s)` / `lower(s)` | Caixa alta / baixa |
| `length(s)` | Comprimento |
| `trim(s)` / `ltrim` / `rtrim` | Remove espaços |
| `substr(s, inicio, n)` | Fatia (índice começa em **1**) |
| `replace(s, de, para)` | Substitui |
| `instr(s, busca)` | Posição (0 se não achar) |
| `printf('%.2f', x)` | Formatação estilo C |

### Número

| Função | Faz |
|--------|-----|
| `ROUND(x, n)` | Arredonda |
| `ABS(x)` | Valor absoluto |
| `CAST(x AS INTEGER)` | Converte (trunca) |
| `MIN(a,b)` / `MAX(a,b)` | Com **dois** argumentos, é escalar (não agregação!) |

### Data

No SQLite as datas são texto ISO. As funções aceitam modificadores.

| Chamada | Devolve |
|---------|---------|
| `date('now')` | `'2026-08-12'` |
| `strftime('%Y-%m', data)` | `'2026-07'` — ✅ o mais útil para agrupar por mês |
| `strftime('%w', data)` | Dia da semana (0=domingo) |
| `date(data, '+30 days')` | Soma dias |
| `julianday(a) - julianday(b)` | Diferença em dias |

In [ ]:
sql("""
SELECT
    nome,
    upper(substr(nome, 1, 1)) || lower(substr(nome, 2)) AS capitalizado,
    length(nome)                                        AS tamanho,
    replace(email, '@email.com', '')                    AS usuario,
    instr(email, '@')                                   AS posicao_arroba
FROM clientes
LIMIT 6
""")

In [ ]:
# ⚠️ MIN/MAX com DOIS argumentos são escalares, não agregações
sql("""
SELECT
    sku,
    preco,
    MIN(preco, 1000)  AS teto_de_1000,
    MAX(preco, 500)   AS piso_de_500
FROM produtos
LIMIT 5
""")

In [ ]:
# Agrupando por mês — o strftime é a chave
sql("""
SELECT
    strftime('%Y-%m', p.data_pedido)               AS mes,
    COUNT(DISTINCT p.id)                           AS pedidos,
    ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS faturamento
FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
WHERE p.status = 'pago'
GROUP BY mes
ORDER BY mes
""")

In [ ]:
# Trabalhando com datas relativas
sql("""
SELECT
    date('now')                        AS hoje,
    date('now', '+30 days')            AS daqui_30_dias,
    date('now', 'start of month')      AS inicio_do_mes,
    date('now', '-1 month')            AS mes_passado,
    strftime('%d/%m/%Y', '2026-07-15') AS formato_br,
    CAST(julianday('2026-08-12') - julianday('2026-07-01') AS INTEGER) AS dias_decorridos
""")

## 🔧 Prática guiada — O dashboard da Aurora em SQL

Aquele relatório que você escreveu em ~200 linhas de Python no Módulo 01. Agora em consultas.

In [ ]:
# ── Indicadores gerais ───────────────────────────────────────
sql("""
SELECT
    COUNT(DISTINCT p.id)                                   AS pedidos_faturados,
    COUNT(DISTINCT p.cliente_id)                           AS clientes_ativos,
    SUM(i.quantidade)                                      AS itens_vendidos,
    ROUND(SUM(i.quantidade * i.preco_unitario), 2)         AS faturamento,
    ROUND(SUM(i.quantidade * i.preco_unitario) / COUNT(DISTINCT p.id), 2) AS ticket_medio
FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
WHERE p.status = 'pago'
""")

In [ ]:
# ── Ranking de praças, com share ─────────────────────────────
sql("""
SELECT
    c.cidade,
    c.uf,
    COUNT(DISTINCT p.id)                           AS pedidos,
    ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS faturamento,
    ROUND(100.0 * SUM(i.quantidade * i.preco_unitario)
          / (SELECT SUM(i2.quantidade * i2.preco_unitario)
             FROM itens_pedido i2
             JOIN pedidos p2 ON p2.id = i2.pedido_id
             WHERE p2.status = 'pago'), 1)         AS share_pct
FROM pedidos p
JOIN clientes c     ON c.id = p.cliente_id
JOIN itens_pedido i ON i.pedido_id = p.id
WHERE p.status = 'pago'
GROUP BY c.cidade, c.uf
ORDER BY faturamento DESC
LIMIT 12
""")

In [ ]:
# ── Top produtos, com margem ─────────────────────────────────
sql("""
SELECT
    pr.sku,
    pr.nome,
    cat.nome                                        AS categoria,
    SUM(i.quantidade)                               AS unidades,
    ROUND(SUM(i.quantidade * i.preco_unitario), 2)  AS receita,
    ROUND(SUM(i.quantidade * (i.preco_unitario - pr.custo)), 2) AS margem,
    ROUND(100.0 * SUM(i.quantidade * (i.preco_unitario - pr.custo))
          / SUM(i.quantidade * i.preco_unitario), 1) AS margem_pct
FROM itens_pedido i
JOIN pedidos    p   ON p.id   = i.pedido_id
JOIN produtos   pr  ON pr.id  = i.produto_id
JOIN categorias cat ON cat.id = pr.categoria_id
WHERE p.status = 'pago'
GROUP BY pr.id, pr.sku, pr.nome, cat.nome
ORDER BY receita DESC
LIMIT 10
""")

In [ ]:
# ── Evolução mensal com composição por canal ─────────────────
sql("""
SELECT
    strftime('%Y-%m', p.data_pedido)  AS mes,
    COUNT(DISTINCT p.id)              AS pedidos,
    ROUND(SUM(CASE WHEN p.canal = 'site'        THEN i.quantidade * i.preco_unitario ELSE 0 END), 2) AS site,
    ROUND(SUM(CASE WHEN p.canal = 'app'         THEN i.quantidade * i.preco_unitario ELSE 0 END), 2) AS app,
    ROUND(SUM(CASE WHEN p.canal = 'marketplace' THEN i.quantidade * i.preco_unitario ELSE 0 END), 2) AS marketplace,
    ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS total
FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
WHERE p.status = 'pago'
GROUP BY mes
ORDER BY mes
""")

In [ ]:
# ── Alerta de estoque, com cobertura em dias ─────────────────
sql("""
SELECT
    pr.sku,
    pr.nome,
    pr.estoque,
    COALESCE(SUM(i.quantidade), 0)                       AS vendidos_3_meses,
    CASE
        WHEN COALESCE(SUM(i.quantidade), 0) = 0 THEN 'sem giro'
        ELSE printf('%.0f dias', pr.estoque / (SUM(i.quantidade) / 90.0))
    END                                                  AS cobertura,
    CASE
        WHEN pr.estoque = 0 THEN '[!] REPOR JA'
        WHEN COALESCE(SUM(i.quantidade), 0) > 0
             AND pr.estoque < SUM(i.quantidade) / 3.0 THEN '[*] atencao'
        ELSE '[ ] ok'
    END                                                  AS alerta
FROM produtos pr
LEFT JOIN itens_pedido i ON i.produto_id = pr.id
LEFT JOIN pedidos p      ON p.id = i.pedido_id AND p.status = 'pago'
GROUP BY pr.id, pr.sku, pr.nome, pr.estoque
ORDER BY pr.estoque
LIMIT 12
""")

## 📝 Exercícios

Escreva a consulta na célula abaixo de cada enunciado.

**E1.** Liste `sku`, `nome` e `preco` dos produtos da categoria 3 (Periféricos) com preço acima de R$ 200, ordenados do mais caro ao mais barato.

**E2.** Quantos clientes existem por segmento? Mostre também o percentual sobre o total.

**E3.** Liste os produtos cujo nome contenha "SSD" **ou** "HD", ignorando maiúsculas/minúsculas.

**E4.** Qual o preço médio, mínimo e máximo por categoria? Inclua o nome da categoria e ordene pelo preço médio decrescente.

**E5.** Quais cidades têm faturamento acima de R$ 40.000 em pedidos pagos? Mostre cidade, UF, nº de pedidos e faturamento.

**E6.** Classifique os clientes em `'sem pedidos'`, `'ocasional'` (1–3), `'recorrente'` (4–7) e `'fiel'` (8+), usando `CASE`. Mostre a contagem de cada classe. *(Dica: precisará de `LEFT JOIN` — veja a aula 03_03 se travar.)*

**E7.** Faturamento por dia da semana. Use `strftime('%w', data_pedido)` e traduza o número para o nome do dia com `CASE`.

**E8.** Quantos pedidos cada canal teve em cada mês? Monte uma tabela pivotada com os meses nas linhas e os canais nas colunas.

**E9.** Liste os 5 produtos com maior **valor imobilizado** (`preco * estoque`) que **não** tiveram nenhuma venda paga.

**E10.** Mostre, para cada UF, o total de pedidos, quantos foram pagos, quantos cancelados, e a taxa de cancelamento em percentual com 1 casa decimal. Ordene pela maior taxa.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

## 📋 Cola de referência

```sql
-- ── Estrutura completa ──
SELECT   DISTINCT coluna, agregacao(x) AS apelido
FROM     tabela
WHERE    condicao                 -- filtra LINHAS (antes de agrupar)
GROUP BY coluna
HAVING   agregacao(x) > 10        -- filtra GRUPOS (depois de agrupar)
ORDER BY apelido DESC
LIMIT    10 OFFSET 20;

-- Ordem de execução: FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT

-- ── Filtros ──
WHERE preco > 100 AND (uf = 'SP' OR uf = 'RJ')
WHERE preco BETWEEN 100 AND 500            -- inclusivo
WHERE uf IN ('SP','RJ','MG')
WHERE nome LIKE 'Note%'                    -- % = n chars, _ = 1 char
WHERE lower(nome) LIKE lower('%busca%')    -- robusto com acentos
WHERE telefone IS NULL                     -- NUNCA use = NULL

-- ── Agregações (ignoram NULL) ──
COUNT(*)                  -- linhas
COUNT(coluna)             -- valores não nulos
COUNT(DISTINCT coluna)
SUM(x)  AVG(x)  MIN(x)  MAX(x)
COALESCE(SUM(x), 0)       -- protege contra NULL sem linhas
GROUP_CONCAT(nome, ', ')

-- ── Condicional ──
CASE WHEN cond THEN a WHEN cond2 THEN b ELSE c END
SUM(CASE WHEN cond THEN 1 ELSE 0 END)      -- conta por condição (pivô)
COALESCE(a, b, c)                          -- primeiro não nulo
NULLIF(a, b)                               -- NULL se a = b (evita divisão por zero)

-- ── Texto ──
upper(s) lower(s) length(s) trim(s)
substr(s, inicio, n)          -- índice começa em 1
replace(s, de, para)  instr(s, busca)
a || b                        -- concatenação
printf('%.2f', x)

-- ── Número ──
ROUND(x, 2)  ABS(x)  CAST(x AS REAL)
x * 1.0 / y                   -- força divisão real

-- ── Data (texto ISO) ──
date('now')
strftime('%Y-%m', data)       -- agrupar por mês
strftime('%w', data)          -- dia da semana (0=dom)
date(data, '+30 days')
julianday(a) - julianday(b)   -- diferença em dias
```

## ✅ Checklist de saída

- [ ] Escrevo `SELECT` com colunas explícitas, não `*`
- [ ] Uso `AS` para nomear expressões
- [ ] Sei que dividir inteiros trunca, e como forçar divisão real
- [ ] Uso `||` para concatenar
- [ ] Domino `WHERE` com `AND`/`OR`/`BETWEEN`/`IN`/`LIKE`
- [ ] **Sempre** ponho parênteses ao misturar `AND` e `OR`
- [ ] Sei que `LIKE` no SQLite ignora caixa só em ASCII
- [ ] Uso `IS NULL`, nunca `= NULL`
- [ ] Sei por que `NOT IN (..., NULL)` devolve zero linhas
- [ ] Recito a ordem lógica: `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`
- [ ] Diferencio `COUNT(*)` de `COUNT(coluna)` de `COUNT(DISTINCT ...)`
- [ ] Sei que agregações ignoram `NULL` e o efeito disso no `AVG`
- [ ] Escrevo `GROUP BY` incluindo todas as colunas não agregadas
- [ ] Sei quando usar `HAVING` e quando usar `WHERE`
- [ ] Uso `COUNT(DISTINCT id)` para não inflar contagens após `JOIN`
- [ ] Domino `CASE WHEN`, inclusive o pivô com `SUM(CASE WHEN ...)`
- [ ] Agrupo por mês com `strftime('%Y-%m', data)`

---

### ➡️ Próxima aula

**`03_03_Joins_e_Subconsultas.ipynb`** — `JOIN`, subconsultas e CTEs. Onde as tabelas separadas voltam a conversar.